# Phase 1: Coordinate Alignment (Two-Stage Homography)
This notebook implements the Self-Refining Georeferencing Pipeline to align the architectural PDF site plan (`bagawat print.pdf`) to the UTM coordinate space of the shapefile footprints (`Buildings_Mask.shp`), without relying on faulty manual GCP files.

The pipeline consists of:
1. **Bootstrap Homography ($H_{init}$)**: Initial low-precision transformation using a few anchor chapel centroids.
2. **RANSAC-Refined Homography ($H_{final}$)**: High-precision transformation using all 263 OCR-extracted chapel labels snapped to shapefile centroids.


In [ ]:
import numpy as np
import cv2
import fitz  # PyMuPDF
import geopandas as gpd
from shapely.geometry import Point
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
import matplotlib.pyplot as plt

# Paths
PDF_PATH = r"../data/BaseSiteCAD/BaseSiteCAD/bagawat print.pdf"
SHP_PATH = r"../data/BaseSiteCAD/130_BuildingFootprintsVectorData/BuildingTracesCurrent/Buildings_Mask.shp"

def rasterize_pdf(pdf_path, dpi=400):
    print("Rasterizing PDF...")
    doc = fitz.open(pdf_path)
    page = doc[0]
    zoom = dpi / 72
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
    img = np.frombuffer(pix.samples, dtype=np.uint8)
    img = img.reshape(pix.height, pix.width, pix.n)
    return img[:, :, :3]  # RGB


In [ ]:
# 1. Load Data
img_rgb = rasterize_pdf(PDF_PATH, dpi=400)
footprints = gpd.read_file(SHP_PATH)
WORKING_CRS = footprints.crs

print(f"Raster shape: {img_rgb.shape}")
print(f"Loaded {len(footprints)} footprints. Working CRS: {WORKING_CRS}")


In [ ]:
# 2. Extract Labels from PDF via OCR
gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
_, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)

config = "--psm 6 -c tessedit_char_whitelist=0123456789"
data = pytesseract.image_to_data(thresh, config=config, output_type=pytesseract.Output.DICT)

ocr_labels = {}
for i, text in enumerate(data["text"]):
    txt = text.strip()
    if txt.isdigit() and int(data["conf"][i]) > 60:
        cx = data["left"][i] + data["width"][i] // 2
        cy = data["top"][i] + data["height"][i] // 2
        if txt not in ocr_labels:
            ocr_labels[txt] = []
        ocr_labels[txt].append((cx, cy))

print(f"Extracted {len(ocr_labels)} unique numeric labels from PDF.")


In [ ]:
# 3. Bootstrap Homography (H_init)
# We pick a few easily identifiable buildings that we know are well separated.
bootstrap_ids = ['23', '24', '25', '26', '175', '210']

manual_pdf_pts = []
manual_utm_pts = []

for b_id in bootstrap_ids:
    if b_id in ocr_labels and len(ocr_labels[b_id]) == 1:
        # Get PDF coordinates
        px, py = ocr_labels[b_id][0]
        # Get UTM centroid from shapefile
        fp = footprints[footprints['ID'] == str(b_id)]
        if not fp.empty:
            cx, cy = fp.iloc[0].geometry.centroid.coords[0]
            manual_pdf_pts.append((px, py))
            manual_utm_pts.append((cx, cy))

src_init = np.array(manual_pdf_pts, dtype=np.float64)
dst_init = np.array(manual_utm_pts, dtype=np.float64)

H_init, _ = cv2.findHomography(src_init, dst_init)
print(f"Computed Bootstrap Homography using {len(src_init)} anchor points.")


In [ ]:
# 4. RANSAC-Refined Homography (H_final)
def pdf_px_to_crs(px, py, H):
    pt = np.array([[px, py, 1.0]], dtype=np.float64)
    mapped = (H @ pt.T).T
    return mapped[0, :2] / mapped[0, 2]

# Map ALL unique OCR labels using H_init, and match to closest footprint centroid
all_pdf_pts = []
all_utm_pts = []

for lbl, pts in ocr_labels.items():
    if len(pts) == 1:
        px, py = pts[0]
        # Transform roughly
        rough_utm = pdf_px_to_crs(px, py, H_init)
        rough_pt = Point(rough_utm[0], rough_utm[1])
        
        # Find nearest footprint
        dists = footprints.geometry.centroid.distance(rough_pt)
        min_idx = dists.idxmin()
        min_dist = dists.min()
        
        # Snap if within 15 meters
        if min_dist < 15.0:
            exact_utm = footprints.loc[min_idx].geometry.centroid.coords[0]
            all_pdf_pts.append((px, py))
            all_utm_pts.append(exact_utm)

src_final = np.array(all_pdf_pts, dtype=np.float64)
dst_final = np.array(all_utm_pts, dtype=np.float64)

H_final, mask = cv2.findHomography(src_final, dst_final, cv2.RANSAC, ransacReprojThreshold=3.0)
inliers = mask.ravel().sum()
print(f"RANSAC Homography fit: {inliers}/{len(src_final)} OCR points used as inliers.")

# RMSE
mapped_pts = []
for p in src_final:
    mapped_pts.append(pdf_px_to_crs(p[0], p[1], H_final))
mapped_pts = np.array(mapped_pts)
rmse = np.sqrt(np.mean(np.sum((mapped_pts - dst_final)**2, axis=1)))
print(f"Final RMSE: {rmse:.3f} meters")


In [ ]:
# 5. Visual Validation
# Overlay the shapefile boundaries on the rasterized PDF using the INVERSE homography
H_inv = np.linalg.inv(H_final)

def crs_to_pdf_px(cx, cy, H_inv):
    pt = np.array([[cx, cy, 1.0]], dtype=np.float64)
    mapped = (H_inv @ pt.T).T
    return mapped[0, :2] / mapped[0, 2]

# Draw polygons on PDF image
img_viz = img_rgb.copy()

for geom in footprints.geometry:
    if geom.geom_type == 'Polygon':
        # get exterior coords
        pts_pdf = [crs_to_pdf_px(x, y, H_inv) for x, y in geom.exterior.coords]
        pts_pdf = np.array(pts_pdf, np.int32).reshape((-1, 1, 2))
        cv2.polylines(img_viz, [pts_pdf], isClosed=True, color=(255, 0, 0), thickness=3)

# Save visualization
out_path = r"../outputs/figures/alignment_validation.jpg"
cv2.imwrite(out_path, cv2.cvtColor(img_viz, cv2.COLOR_RGB2BGR))

plt.figure(figsize=(12, 12))
plt.imshow(img_viz)
plt.title("Shapefile Footprints Overlaid on PDF")
plt.axis("off")
plt.show()
print(f"Validation image saved to {out_path}")
